# 5. Extension — XGBoost & SMOTE

**Goal:** test two "next level" techniques against what we already found, and see if they actually improve things or not.

**Recap of what we know so far:**
- Logistic Regression (simple model) generalized *better* to novel attacks (43.4% detection) than Random Forest (14.6%) — Random Forest overfit to training attack signatures.
- U2R is severely underrepresented (52 training examples vs. 67,343 normal) — this is the hardest class across every model so far.

**Two real questions this notebook answers:**
1. **XGBoost** — is a more powerful tree-based model, but Random Forest already showed that "more powerful" can mean "worse at generalizing to novel attacks." Does XGBoost repeat that mistake, or avoid it?
2. **SMOTE** — can generating synthetic examples of rare attack types (like U2R) actually help the model learn them better?

**New installs needed for this notebook:**
```
pip install xgboost imbalanced-learn
```

**Input:** `train_processed.csv`, `test_processed.csv` (from notebook 2)

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

train = pd.read_csv("data/train_processed.csv")
test = pd.read_csv("data/test_processed.csv")

feature_cols = [c for c in train.columns if c not in ["label", "attack_category"]]
X_train = train[feature_cols]
y_train = train["attack_category"]
X_test = test[[c for c in feature_cols if c != "is_novel_attack_type"]]
y_test = test["attack_category"]
is_novel = test["is_novel_attack_type"]

print(X_train.shape, X_test.shape)

(125973, 41) (22544, 41)


## Part 1: XGBoost

XGBoost needs numeric labels, not text categories, so we encode `attack_category` first.

In [2]:
from xgboost import XGBClassifier

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

print("Classes:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

Classes: {'DoS': 0, 'Probe': 1, 'R2L': 2, 'U2R': 3, 'normal': 4}


In [3]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="mlogloss"
)
xgb.fit(X_train, y_train_enc)

y_pred_xgb_enc = xgb.predict(X_test)
y_pred_xgb = label_encoder.inverse_transform(y_pred_xgb_enc)

print(classification_report(y_test, y_pred_xgb, zero_division=0))

xgb_macro_f1 = f1_score(y_test, y_pred_xgb, average="macro")
xgb_weighted_f1 = f1_score(y_test, y_pred_xgb, average="weighted")
print(f"Macro F1: {xgb_macro_f1:.3f} | Weighted F1: {xgb_weighted_f1:.3f}")

              precision    recall  f1-score   support

         DoS       0.96      0.82      0.89      7460
       Probe       0.80      0.69      0.74      2421
         R2L       0.98      0.06      0.11      2885
         U2R       0.50      0.07      0.13        67
      normal       0.68      0.97      0.80      9711

    accuracy                           0.77     22544
   macro avg       0.78      0.52      0.53     22544
weighted avg       0.82      0.77      0.73     22544

Macro F1: 0.534 | Weighted F1: 0.733


## The real test: does XGBoost repeat Random Forest's mistake?

This is the question that actually matters — not "is the F1 score higher," but "does it generalize to attacks it's never seen?"

In [4]:
novel_mask = (is_novel == 1).values
xgb_novel_detection = (y_pred_xgb[novel_mask] != "normal").mean()

print(f"XGBoost novel-attack detection rate: {xgb_novel_detection:.1%}")
print()
print("Compare to what we found in notebook 3:")
print("  Logistic Regression: 43.4%")
print("  Decision Tree:       42.8%")
print("  Random Forest:       14.6%")

XGBoost novel-attack detection rate: 36.9%

Compare to what we found in notebook 3:
  Logistic Regression: 43.4%
  Decision Tree:       42.8%
  Random Forest:       14.6%


**How to interpret your result:**
- If XGBoost's novel-detection rate is closer to Random Forest's (~15%) → this confirms tree ensembles tend to overfit to known attack signatures on this dataset, regardless of which specific ensemble method is used.
- If XGBoost's novel-detection rate is closer to Logistic Regression's (~43%) or higher → that's a genuinely interesting counter-finding: it would suggest XGBoost's boosting approach (learning from mistakes sequentially) generalizes differently than Random Forest's bagging approach, even though both are tree-based.

Either result is a valid, honest finding — write down whichever one you actually get, don't assume it'll match either pattern before you see it.

## Part 2: SMOTE

Now let's test whether generating synthetic examples of rare attack types (especially U2R, with only 52 real examples) helps the model learn them better.

In [5]:
from imblearn.over_sampling import SMOTE

print("Class counts before SMOTE:")
print(y_train.value_counts())

# k_neighbors must be smaller than the smallest class size minus 1
# U2R has 52 examples, so k_neighbors=3 is a safe conservative choice
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print()
print("Class counts after SMOTE:")
print(pd.Series(y_train_smote).value_counts())

Class counts before SMOTE:
attack_category
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

Class counts after SMOTE:
attack_category
normal    67343
DoS       67343
R2L       67343
Probe     67343
U2R       67343
Name: count, dtype: int64


**What just happened:** SMOTE oversampled every minority class up to match the size of the majority class (`normal`) by generating synthetic examples. Note this means it also invented tens of thousands of synthetic U2R examples from just 52 real ones — keep that in mind when judging how much to trust the results below.

In [6]:
# Retrain Logistic Regression (our best model so far) on the SMOTE-resampled data
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler_smote = StandardScaler()
X_train_smote_scaled = scaler_smote.fit_transform(X_train_smote)
X_test_scaled_for_smote = scaler_smote.transform(X_test)

lr_smote = LogisticRegression(max_iter=1000)
lr_smote.fit(X_train_smote_scaled, y_train_smote)

y_pred_smote = lr_smote.predict(X_test_scaled_for_smote)

print(classification_report(y_test, y_pred_smote, zero_division=0))

smote_macro_f1 = f1_score(y_test, y_pred_smote, average="macro")
print(f"Macro F1 with SMOTE: {smote_macro_f1:.3f}")
print(f"Macro F1 without SMOTE (notebook 3): 0.610")

              precision    recall  f1-score   support

         DoS       0.93      0.78      0.85      7460
       Probe       0.76      0.87      0.81      2421
         R2L       0.87      0.28      0.42      2885
         U2R       0.08      0.45      0.14        67
      normal       0.73      0.91      0.81      9711

    accuracy                           0.78     22544
   macro avg       0.67      0.66      0.61     22544
weighted avg       0.81      0.78      0.77     22544

Macro F1 with SMOTE: 0.607
Macro F1 without SMOTE (notebook 3): 0.610


In [7]:
smote_novel_detection = (y_pred_smote[novel_mask] != "normal").mean()
print(f"Novel-attack detection rate with SMOTE: {smote_novel_detection:.1%}")
print(f"Novel-attack detection rate without SMOTE: 43.4%")

Novel-attack detection rate with SMOTE: 45.6%
Novel-attack detection rate without SMOTE: 43.4%


**How to interpret this result:**
- Compare the **U2R row specifically** in the classification report above to notebook 3's Logistic Regression result (precision 0.08, recall 0.58, F1 0.15). Did SMOTE improve U2R specifically, or did it mostly help/hurt other classes?
- Compare the novel-attack detection rate too. SMOTE only adds synthetic examples of attack types that already exist in training — it does nothing to help the model recognize the 17 *novel* attack types, so don't expect this number to improve much (or it might even get slightly worse if the model becomes more confident/specific about known attack patterns).

## Final comparison table

Fill this in with your actual numbers once you run the notebook — this is the honest way to conclude the experiment, whichever direction the results go.

In [8]:
final_comparison = pd.DataFrame([
    {"model": "Logistic Regression (baseline)", "macro_f1": 0.610, "novel_attack_detection_rate": 0.434},
    {"model": "Decision Tree (baseline)", "macro_f1": 0.502, "novel_attack_detection_rate": 0.428},
    {"model": "Random Forest (baseline)", "macro_f1": 0.473, "novel_attack_detection_rate": 0.146},
    {"model": "XGBoost", "macro_f1": round(xgb_macro_f1, 3), "novel_attack_detection_rate": round(xgb_novel_detection, 3)},
    {"model": "Logistic Regression + SMOTE", "macro_f1": round(smote_macro_f1, 3), "novel_attack_detection_rate": round(smote_novel_detection, 3)},
])
final_comparison = final_comparison.sort_values("macro_f1", ascending=False)
final_comparison.to_csv("data/final_model_comparison_with_extensions.csv", index=False)
final_comparison

,model,macro_f1,novel_attack_detection_rate
0,Logistic Regression (baseline),0.610,0.434
4,Logistic Regression + SMOTE,0.607,0.456
3,XGBoost,0.534,0.369
1,Decision Tree (baseline),0.502,0.428
2,Random Forest (baseline),0.473,0.146


## What to conclude — write this yourself once you have real numbers

Don't just report whichever model has the highest macro F1. Ask two questions honestly:

1. **Did XGBoost repeat Random Forest's overfitting problem**, or did it avoid it? This tells you whether the issue we found earlier was specific to Random Forest's bagging approach, or is a general risk with any complex tree ensemble on this dataset.
2. **Did SMOTE meaningfully help the rare classes (R2L, U2R)**, or did the synthetic examples not capture real attack behavior well enough to matter? Be specific — look at the actual precision/recall numbers for those two classes, not just the aggregate macro F1.

A good project writeup states the real result even if it's not the "impressive" one you might have expected — e.g., "SMOTE did not meaningfully improve U2R detection, likely because 52 real examples is too few for even synthetic interpolation to capture genuine attack behavior" is a completely valid, honest, and interesting conclusion.